<a href="https://colab.research.google.com/github/Paulo83-dev/mestrado-computacao-aplicada/blob/main/machine-learning/aula05_árvore_de_decisão_atributos_discretos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌳 Aula 05: Árvore de Decisão do Zero (Atributos Discretos)

Nesta aula prática, vamos construir uma **Árvore de Decisão do zero** em Python, utilizando como base a biblioteca Scikit-Learn (criando um estimador customizado compatível com suas APIs).

A base utilizada será o clássico dataset de avaliação de carros (**Car Evaluation**) do repositório UCI. Todos os atributos desse dataset são **discretos/categóricos**, o que facilita nossa primeira implementação recursiva.

### 📦 Instalando a biblioteca do UCI ML Repository

#### 🔍 O que este bloco faz?
Instala o pacote `ucimlrepo`, que permite baixar bases de dados públicas diretamente do repositório oficial da UCI.

In [1]:
!pip install ucimlrepo -q

### 📥 Carregando os Dados (Car Evaluation)

#### 🔍 O que este bloco faz?
Baixa a base de dados sob ID 19 (Car Evaluation) e separa os atributos (`X`) e as respostas (`y`).

#### 🎯 Qual a intenção pedagógica?
Mostrar como acessar e ler metadados de datasets clássicos. Veja na saída a descrição de cada atributo categórico, como o preço de compra (`buying`) e nível de segurança estimado (`safety`).

In [6]:
from ucimlrepo import fetch_ucirepo

# Busca a base Car Evaluation da UCI
car_evaluation = fetch_ucirepo(id=19)

X = car_evaluation.data.features.to_numpy()
y = car_evaluation.data.targets.to_numpy()[:,0]

print(car_evaluation.variables)

       name     role         type demographic  \
0    buying  Feature  Categorical        None   
1     maint  Feature  Categorical        None   
2     doors  Feature  Categorical        None   
3   persons  Feature  Categorical        None   
4  lug_boot  Feature  Categorical        None   
5    safety  Feature  Categorical        None   
6     class   Target  Categorical        None   

                                         description units missing_values  
0                                       buying price  None             no  
1                           price of the maintenance  None             no  
2                                    number of doors  None             no  
3              capacity in terms of persons to carry  None             no  
4                           the size of luggage boot  None             no  
5                        estimated safety of the car  None             no  
6  evaulation level (unacceptable, acceptable, go...  None             no  

In [8]:
for i in range(X.shape[1]):
  values = set(X[:,i])
  print(f"{i}. {car_evaluation.variables['name'][i]}:\t{values}")

0. buying:	{'high', 'vhigh', 'med', 'low'}
1. maint:	{'high', 'vhigh', 'med', 'low'}
2. doors:	{'5more', '3', '4', '2'}
3. persons:	{'4', '2', 'more'}
4. lug_boot:	{'big', 'small', 'med'}
5. safety:	{'high', 'low', 'med'}


### 📊 Classes de Destino (Target)

#### 🔍 O que este bloco faz?
Mapeia as classes possíveis do nosso target (`y`).

In [9]:
set(y)

{'acc', 'good', 'unacc', 'vgood'}

### 📊 Classes de Destino (Target)

#### 🔍 O que este bloco faz?
Mapeia as classes possíveis do nosso target (`y`).

In [18]:
for label in set(y):
  print(f"{label}:\t{100*sum(y==label)/len(y):.4}%")

vgood:	3.762%
good:	3.993%
unacc:	70.02%
acc:	22.22%


### 📊 Classes de Destino (Target)

#### 🔍 O que este bloco faz?
Mapeia as classes possíveis do nosso target (`y`).

In [77]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score

def most_common(y):
  return max(set(y), key=lambda label: sum(y==label))

class ZeroR(BaseEstimator, ClassifierMixin):
  def fit(self, X, y):
    self.answer = most_common(y)
  def predict(self, X):
    return [self.answer]*X.shape[0]

model = ZeroR()
model.fit(X, y)
y_pred = model.predict(X)
print(accuracy_score(y, y_pred))

0.7002314814814815


### ✂️ Divisão Treino e Teste com Estratificação

#### 🔍 O que este bloco faz?
Divide os dados em treino e teste com estratificação baseada nas classes do target.

#### 🎯 Qual a intenção pedagógica?
A estratificação (`stratify=y`) garante que a proporção de 70% da classe majoritária seja idêntica tanto no treino quanto no teste, evitando viés de amostragem na avaliação.

In [78]:
from sklearn.model_selection import train_test_split

# Split treino/teste estratificado
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    stratify=y,
                                                    shuffle=True,
                                                    random_state=42)
model = ZeroR()
model.fit(X_train, y_train)
yhat = model.predict(X_test)
accuracy = accuracy_score(y_test, yhat)
print('Acurácia ZeroR no Teste: %.3f%%' % (accuracy*100))

Accuracy: 69.942


### 🎲 Árvore de Decisão com Splits Aleatórios

#### 🔍 O que este bloco faz?
Implementa uma árvore de decisão recursiva simples, onde a característica (`self.feature`) e o valor de corte (`self.value`) são escolhidos aleatoriamente em cada nó.

#### 🎯 Qual a intenção pedagógica?
Compreender a estrutura recursiva clássica das árvores: um nó raiz faz uma pergunta (`X[:, self.feature] == self.value`) e cria duas sub-árvores recursivamente (`self.equals_tree` e `self.not_equals_tree`) até atingir folhas onde não há mais dados para dividir (nó folha).

In [138]:
import numpy as np

# Árvore de Decisão com decisões de corte aleatórias
class DecisionTree(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.feature = np.random.randint(X.shape[1])
        self.value = np.random.choice(list(set(X[:, self.feature])))
        equals = X[:, self.feature] == self.value
        if sum(equals) > 0 and sum(~equals) > 0:
            self.equals_tree = DecisionTree().fit(X[equals], y[equals])
            self.not_equals_tree = DecisionTree().fit(X[~equals], y[~equals])
        else:
            self.answer = most_common(y)
        return self

    def predict(self, X):
        if hasattr(self, 'answer'):
            return [self.answer]*X.shape[0]
        else:
            equals = X[:, self.feature] == self.value
            return np.where(equals, self.equals_tree.predict(X), self.not_equals_tree.predict(X))

model = DecisionTree()
model.fit(X, y)
ypred = model.predict(X)
print("Acurácia da Árvore Aleatória:", accuracy_score(y, ypred))

0.7563657407407407


### 📊 Classes de Destino (Target)

#### 🔍 O que este bloco faz?
Mapeia as classes possíveis do nosso target (`y`).

In [140]:
def gini(y):
  labels = list(set(y))
  x = 0
  for label in labels:
    label_prob = np.mean(y==label)
    x += label_prob**2
  return 1-x

print(gini(y_train))

0.4570454112310227


### 💡 Exemplos de Cálculo de Gini

#### 🔍 O que este bloco faz?
Calcula o Gini para um caso perfeitamente puro (todos do mesmo grupo) e um caso de máxima impureza (todos de grupos diferentes).

#### 🎯 Qual a intenção pedagógica?
Verificar na prática que um nó homogêneo resulta em Gini 0.0 (pureza total) e um nó heterogêneo resulta em Gini próximo de 1.0.

In [144]:
print("Gini pura:", gini(np.ones(100)))
print("Gini impura:", gini(np.arange(100)))

0.0


In [146]:
print(gini(np.arange(100)))

0.99


### 🔀 Medindo a Impureza do Split

#### 🔍 O que este bloco faz?
Implementa a função `impurity_value`, que simula uma divisão binária $x == value$ e calcula a impureza ponderada combinando as duas partições resultantes.

#### 🎯 Qual a intenção pedagógica?
Entender que o custo de um split é a soma ponderada das impurezas de suas folhas descendentes.

In [151]:
# Função de cálculo de impureza média pós-split
def impurity_value(x, y, value, impurity_function):
  equals = x == value
  equals_impurity = impurity_function(y[equals])
  not_equals_impurity = impurity_function(y[~equals])
  return (np.mean(equals)) * equals_impurity + (np.mean(~equals)) * not_equals_impurity

print("Impureza para split no buying='vhigh':", impurity_value(X_train[:,0], y_train, "vhigh", gini))

0.4482135562918824


### 🎯 Buscando o Melhor Ponto de Corte (Best Split) para um Atributo

#### 🔍 O que este bloco faz?
Varre todos os valores únicos de um atributo e encontra aquele que gera o menor valor de impureza Gini pós-split.

#### 🎯 Qual a intenção pedagógica?
Mostrar o algoritmo de força-bruta local: testamos todas as perguntas lógicas possíveis para uma característica específica e selecionamos a melhor pergunta.

In [162]:
# Busca do melhor ponto de corte para um atributo
def best_split(x, y, impurity_function):
  best_value = None
  best_value_impurity = float('inf')
  for value in set(x):
    value_impurity = impurity_value(x, y, value, impurity_function)
    if value_impurity < best_value_impurity:
      best_value = value
      best_value_impurity = value_impurity
  return best_value, best_value_impurity

print("Melhor valor e impureza para portas:", best_split(X_train[:,2], y_train, gini))

('5more', np.float64(0.45634557119913965))


### 🏆 Buscando a Melhor Característica (Best Feature)

#### 🔍 O que este bloco faz?
Avalia o melhor split de todas as características disponíveis e seleciona aquela que resulta no menor Gini geral.

#### 🎯 Qual a intenção pedagógica?
Entender a decisão gananciosa (greedy choice) que a árvore faz em cada nó: ela varre todos os caminhos e escolhe a divisão ótima naquele instante.

In [166]:
# Busca da melhor característica e melhor ponto de corte
def best_feature(X, y, impurity_function):
  best_feature = None
  best_value = None
  best_value_impurity = float('inf')
  for feature in range(X.shape[1]):
    value, _ = best_split(X[:,feature], y, impurity_function)
    feature_impurity = impurity_value(X[:,feature], y, value, impurity_function)
    if feature_impurity < best_value_impurity:
      best_feature = feature
      best_value_impurity = feature_impurity
      best_value = value
  return best_feature, best_value, best_value_impurity

print("Melhor feature geral:", best_feature(X_train, y_train, gini))

(5, 'low', np.float64(0.38499511557696947))


### 🛠️ Árvore de Decisão Completa (Construção Inteligente)

#### 🔍 O que este bloco faz?
Combina as funções de busca do melhor split (`best_feature`) na estrutura recursiva da árvore de decisão.

#### 🎯 Qual a intenção pedagógica?
Unir a recursão lógica com a seleção baseada na impureza de Gini. Esta implementação cresce até atingir nós purificados completos.

In [167]:
# Árvore de Decisão baseada na impureza de Gini
class DecisionTree(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.feature, self.value, self.impurity = best_feature(X, y, gini)
        equals = X[:, self.feature] == self.value
        if sum(equals) > 0 and sum(~equals) > 0:
            self.equals_tree = DecisionTree().fit(X[equals], y[equals])
            self.not_equals_tree = DecisionTree().fit(X[~equals], y[~equals])
        else:
            self.answer = most_common(y)
        return self

    def predict(self, X):
        if hasattr(self, 'answer'):
            return [self.answer]*X.shape[0]
        else:
            equals = X[:, self.feature] == self.value
            return np.where(equals, self.equals_tree.predict(X), self.not_equals_tree.predict(X))

model = DecisionTree()
model.fit(X, y)
ypred = model.predict(X)
print("Acurácia de treino:", accuracy_score(y, ypred))

1.0


### 📈 Avaliação da Árvore de Decisão Completa

#### 🔍 O que este bloco faz?
Mede a acurácia de treinamento e no conjunto de teste.

#### 🎯 Qual a intenção pedagógica?
Notar que o treino obteve **acurácia de 1.0 (100%)**. Árvores completas sem podas tendem a memorizar perfeitamente o treino, o que pode causar overfitting. No entanto, a acurácia no teste foi excelente (~96.8%).

In [168]:
# Treino no conjunto de treino e avaliação no conjunto de teste
model = DecisionTree()
model.fit(X_train, y_train)
ypred = model.predict(X_test)
print("Acurácia no teste:", accuracy_score(y_test, ypred))

0.9682080924855492


### 🔄 Validação Cruzada da Árvore Completa

#### 🔍 O que este bloco faz?
Executa uma validação cruzada K-Fold para estimar estatisticamente o desempenho geral do nosso classificador customizado.

In [172]:
from sklearn.model_selection import cross_val_score, KFold

# Validação cruzada da nossa Árvore Completa
scores = cross_val_score(DecisionTree(), X, y, cv=KFold(n_splits=5, shuffle=True))
print("Pontuações:", scores)
print("Média:", np.mean(scores))

[0.97687861 0.98554913 0.96242775 0.96811594 0.96811594]
0.9722174750774901


### ✂️ Controlando o Overfitting: Profundidade Máxima (max_depth)

#### 🔍 O que este bloco faz?
Implementa a restrição de profundidade máxima (`max_depth`). A recursão para quando atingimos o limite de níveis.

#### 🎯 Qual a intenção pedagógica?
Demonstrar uma das formas clássicas de **pré-poda (pre-pruning)**. Limitar a profundidade impede que a árvore se especialize demais em detalhes e ruídos do conjunto de treino.

In [182]:
# Árvore de Decisão com restrição de profundidade máxima
class DecisionTree(BaseEstimator, ClassifierMixin):
    def __init__(self, max_depth=9999999):
        self.max_depth = max_depth

    def fit(self, X, y):
        self.feature, self.value, self.impurity = best_feature(X, y, gini)
        equals = X[:, self.feature] == self.value
        if sum(equals) > 0 and sum(~equals) > 0 and self.max_depth>0:
            self.equals_tree = DecisionTree(self.max_depth-1).fit(X[equals], y[equals])
            self.not_equals_tree = DecisionTree(self.max_depth-1).fit(X[~equals], y[~equals])
        else:
            self.answer = most_common(y)
        return self

    def predict(self, X):
        if hasattr(self, 'answer'):
            return [self.answer]*X.shape[0]
        else:
            equals = X[:, self.feature] == self.value
            return np.where(equals, self.equals_tree.predict(X), self.not_equals_tree.predict(X))

model = DecisionTree(5)
scores = cross_val_score(model, X, y, cv=KFold(n_splits=5, shuffle=True))
print("Acurácias com max_depth=5:", scores)
print("Média:", np.mean(scores))

[0.86127168 0.83815029 0.88150289 0.84057971 0.91014493]
0.8663298986344976


### ✂️ Controlando o Overfitting: Divisão Mínima de Amostras (min_sample_split)

#### 🔍 O que este bloco faz?
Adiciona o critério `min_sample_split`: só permitimos a recursão caso ambos os subgrupos gerados contenham mais amostras do que o mínimo configurado.

#### 🎯 Qual a intenção pedagógica?
Mostrar mais uma técnica comum de pré-poda. Impedir splits que isolem muito poucas amostras garante que as folhas representem tendências gerais dos dados e não amostras individuais barulhentas.

In [192]:
# Árvore de Decisão com limite de profundidade e tamanho mínimo do split
class DecisionTree(BaseEstimator, ClassifierMixin):
    def __init__(self, max_depth=9999999, min_sample_split=2):
        self.max_depth = max_depth
        self.min_sample_split = min_sample_split

    def fit(self, X, y):
        self.feature, self.value, self.impurity = best_feature(X, y, gini)
        equals = X[:, self.feature] == self.value
        if sum(equals) > self.min_sample_split and \
           sum(~equals) > self.min_sample_split and \
           self.max_depth>0:
            self.equals_tree = DecisionTree(self.max_depth-1).fit(X[equals], y[equals])
            self.not_equals_tree = DecisionTree(self.max_depth-1).fit(X[~equals], y[~equals])
        else:
            self.answer = most_common(y)
        return self

    def predict(self, X):
        if hasattr(self, 'answer'):
            return [self.answer]*X.shape[0]
        else:
            equals = X[:, self.feature] == self.value
            return np.where(equals, self.equals_tree.predict(X), self.not_equals_tree.predict(X))

model = DecisionTree(20, 10)
scores = cross_val_score(model, X, y, cv=KFold(n_splits=5, shuffle=True))
print("Acurácias com max_depth=20 e min_sample_split=10:", scores)
print("Média:", np.mean(scores))

[0.9566474  0.95375723 0.94219653 0.97101449 0.95652174]
0.9560274775906844
